# 05. Food outlets from the establishment register

This notebook turns the national directory of economic units into the sets of
food destinations that walking distances are measured to.

A measure of food access has to decide which outlets count as a source of food,
and that decision determines the result. Rather than adopt one definition, this
analysis builds several overlapping sets and reports the measured access under
each, so that the sensitivity of the answer to the definition is part of what is
reported.

The sets form two branches and a union.

The self-service branch begins with supermarkets, which are full format
self-service retail. Warehouse clubs are added next, because the food desert
definitions developed in the United States and applied elsewhere count
supercentres and club stores as supermarkets. Small format chain self-service
retail, which in Mexico is mostly convenience chains, is added after that.
National household expenditure data place these chains at about two per cent of
metropolitan food spending against about fifteen per cent for supermarkets, so
they are a separate step rather than part of the supermarket set. Pharmacies
holding a grocery section complete the branch.

The traditional branch begins with the sellers of perishable produce, protein and
dairy: butchers, poulterers, fishmongers, greengrocers and dairy shops. The
specialist fresh and staple set adds sellers of grains and seeds, tortillerías
and traditional bakeries. Tortilla making and traditional bread baking are
classified under food manufacturing rather than retail, because the product is
made on the premises, so they are drawn from the manufacturing codes: nixtamal
milling with maize tortillas, traditional wheat flour tortillas, and traditional
bakery. Industrial bakery and the manufacture of biscuits and pasta supply other
retailers rather than households at a counter, and are left out. The staple
trades are close to a third of the fresh and staple set, which is why the
perishable set is built beside it. Two extensions of the fresh and staple set are
built, and neither contains the other: one adds craft dairy producers, which the
directory also files under manufacturing, and one adds neighbourhood grocery
stores, which are close to ubiquitous in Mexican cities.

The union is every establishment in any of those sets together with the
remaining food retail and food service trades. Food service is retained because
low cost prepared food is a real channel through which households obtain meals,
and it enters only this widest set because eating out is not the same activity
as provisioning. Bars, cantinas and nightclubs are establishments whose main
trade is drink, so they are reported but enter no set. Remaining food retail is
confectionery and baking supplies, ice cream, other foods, and non-alcoholic
drinks and ice, together with the trades selling those same goods that the
directory places under manufacturing: ice cream and ice lolly making, water
purification and bottling, and ice making. A trade counts the same way whichever
sector the directory files it under, since a household buying at the counter
cannot tell the difference. Department stores and alcohol and tobacco retail are
reported but enter no set.

Street commerce is a further category, recorded by the census street audit at
block level rather than in this directory, and it joins the traditional branch
and the union at the routing stage.

Every set is defined by activity code in a single dictionary. The codes present
in the extracts are printed with counts before any classification is applied,
restricted to retail, food service and the manufacture of food and beverages,
since no other part of manufacturing sells food to households.

Coordinates are published for every establishment, and some of them are the
centroid of a municipality or a locality rather than the position of the
establishment. Such a coordinate passes a test on the national extent
perfectly, so the establishments sharing each coordinate are counted, the
largest clusters are listed with the trades they belong to, and each record
carries a flag for whether it sits on a point shared with others. A cluster of
destinations on one point puts every block near that point at the wrong
distance.

Output is `Results/05_outlets.parquet`, one row per establishment with its
projected coordinates, its metropolitan area, the shared point flag, one column
per layer and one column per destination set.

Runtime is a few minutes.

## 1. Setup

In [ ]:
import sys, os, glob, importlib

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = os.environ.get('MXO_BASE', '/content/drive/MyDrive/Mexico_Obesity')
os.environ['MXO_BASE'] = PROJECT
sys.path.insert(0, f'{PROJECT}/Code')
import mxo_utils as mx
importlib.reload(mx)

assert mx.BASE == PROJECT, f'the module resolves to {mx.BASE}'

import numpy as np
import pandas as pd
import geopandas as gpd

rep = mx.Report('05_food_outlets').start()
mx.flow_reset()

INPUTS = mx.DIRS['inputs']
METRO = gpd.read_file(os.path.join(mx.DIRS['results'],
                                   '03_metro_boundaries.gpkg'),
                      layer='metro_boundaries')
print(f'{len(METRO)} metropolitan boundaries, projection {METRO.crs}')

## 2. Reading the directory

The extracts cover the retail food codes and food service. Only the variables
needed to locate and classify an establishment are read. Coordinates are
published in degrees and are projected to metres before anything spatial is
done with them.

In [ ]:
USE = ['id', 'nom_estab', 'raz_social', 'codigo_act', 'nombre_act', 'per_ocu',
       'cve_ent', 'cve_mun', 'cve_loc', 'ageb', 'manzana',
       'latitud', 'longitud']

files = sorted(glob.glob(os.path.join(INPUTS, 'DENUE', '*.csv')))
assert files, f'no extracts found in {INPUTS}/DENUE'
print(f'{len(files)} extracts')

parts = []
for f in files:
    with mx.stage(os.path.basename(f)):
        df = pd.read_csv(f, usecols=lambda col: col in USE, dtype=str,
                         encoding='latin1', low_memory=False)
        df['src'] = os.path.basename(f)
        parts.append(df)

D = pd.concat(parts, ignore_index=True)
D['codigo_act'] = D['codigo_act'].astype(str).str.strip()
D['latitud'] = pd.to_numeric(D['latitud'], errors='coerce')
D['longitud'] = pd.to_numeric(D['longitud'], errors='coerce')

print(f'\n{len(D):,} establishment records')
mx.flow('records in the extracts', len(D), 'all files')

## 3. The activity codes present

The codes under which the directory files these establishments are listed with
their counts and their published descriptions, so that the classification below
can be read against the contents of the data. The listing covers retail trade,
food and beverage service, and the manufacture of food and beverages. Records
outside those sectors are counted and enter no set.

In [ ]:
SECTORS = ('46', '72', '311', '312')
in_scope = D['codigo_act'].str.startswith(SECTORS)

CODES = (D.loc[in_scope].groupby(['codigo_act', 'nombre_act']).size()
           .rename('records').reset_index()
           .sort_values('records', ascending=False))
print(f'{len(CODES)} codes in retail, food service and food manufacturing')
print(f'{int((~in_scope).sum()):,} records outside those sectors\n')
pd.set_option('display.max_colwidth', 90)
print(CODES.to_string(index=False))
rep.add_table('activity codes in the food sectors', CODES, max_rows=150)

## 4. Classification

Each layer is defined by activity code. The layers do not overlap, and the
destination sets below are unions of layers, so a layer may belong to several
sets.

In [ ]:
LAYERS = {
    # Full format self-service retail. Warehouse clubs share this code and are
    # identified separately by establishment name below.
    'supermarket':     {'exact': ['462111']},

    # Small format chain self-service retail.
    'minisuper':       {'exact': ['462112']},

    # Pharmacies holding a grocery section. The last step of the self-service
    # branch, and part of the widest set.
    'pharmacy_mini':   {'exact': ['464112']},

    # Specialist sellers of fresh food, and the staple trades the directory
    # places under food manufacturing. Industrial bakery, 311811, and the
    # manufacture of biscuits and pasta, 311820, supply other retailers rather
    # than households at a counter and are excluded.
    'fresh':           {'exact': ['461121',    # red meat
                                  '461122',    # poultry
                                  '461123',    # fish and seafood
                                  '461130',    # fruit and vegetables
                                  '461140',    # grains, seeds and dried chillies
                                  '461150',    # milk, dairy and cold cuts
                                  '311812',    # traditional bakery
                                  '311813',    # traditional wheat tortillas
                                  '311830']},  # maize tortillas, nixtamal milling

    # Craft dairy, which the directory places under manufacturing. It extends
    # the fresh set in one direction and is part of the widest set.
    'dairy_craft':     {'exact': ['311513']},

    # Neighbourhood grocery stores.
    'abarrotes':       {'exact': ['461110']},

    # Remaining food retail and the manufacturing codes covering the same
    # trades, entering only the widest set.
    'other_food':      {'exact': ['461160',    # confectionery and baking supplies
                                  '461170',    # ice cream and ice lollies
                                  '461190',    # other foods
                                  '461213',    # non-alcoholic drinks and ice
                                  '311520',    # ice cream and ice lolly making
                                  '312112',    # water purification and bottling
                                  '312113']},  # ice making

    # Prepared food for immediate consumption, entering only the widest set.
    # Bars, cantinas and nightclubs belong to the same sector and are kept
    # apart below.
    'prepared':        {'prefix': ['722'], 'exclude': ['722411', '722412']},

    # Bars and cantinas, 722412, and nightclubs, 722411. Reported and entering
    # no set.
    'drinking':        {'exact': ['722411', '722412']},

    # Reported and entering no set.
    'department':      {'exact': ['462210']},
    'alcohol_tobacco': {'exact': ['461211', '461212', '461220']},
}

# The part of the fresh set that sells perishable produce, protein and dairy,
# without the staple trades and without seeds and grains.
PERISHABLE_CODES = ['461121', '461122', '461123', '461130', '461150']

# Layers whose union is the widest definition. Every destination set below is
# contained in this union.
FOOD_LAYERS = ['supermarket', 'minisuper', 'pharmacy_mini', 'fresh',
               'dairy_craft', 'abarrotes', 'other_food', 'prepared']

# The staple trades the fresh set is defined to include, all of which the
# directory files under manufacturing.
STAPLE_CODES = ['311812', '311813', '311830']

# Establishment names identifying a warehouse club, matched on a form of the
# name with accents and punctuation removed.
CLUB_NAMES = ['COSTCO', 'SAMS CLUB', 'SAM S CLUB', 'CITY CLUB']


def code_mask(spec, codes):
    m = pd.Series(False, index=codes.index)
    for e in spec.get('exact', []):
        m |= codes.eq(e)
    for p in spec.get('prefix', []):
        m |= codes.str.startswith(p)
    for e in spec.get('exclude', []):
        m &= ~codes.eq(e)
    return m


for layer, spec in LAYERS.items():
    D[layer] = code_mask(spec, D['codigo_act'])

# The layers partition the classified records, so no record may carry two.
overlap = int((D[list(LAYERS)].sum(axis=1) > 1).sum())
assert overlap == 0, f'{overlap} records fall in more than one layer'

name = mx.normalise_text(D['nom_estab'].fillna('') + ' ' +
                         D['raz_social'].fillna(''))
D['club'] = D['supermarket'] & name.str.contains('|'.join(CLUB_NAMES), regex=True)

print('records per layer:\n')
for layer in list(LAYERS) + ['club']:
    print(f'   {layer:16s} {int(D[layer].sum()):>9,}')

print('\nbars, cantinas and nightclubs, reported and entering no set:\n')
print(D.loc[D.drinking].groupby(['codigo_act', 'nombre_act']).size()
       .rename('records').reset_index().to_string(index=False))

### The manufacturing codes the sets draw on

The fresh set and the remaining food retail set each draw on codes the
directory files under manufacturing, so every food and beverage manufacturing
code found is listed with the trades it covers. A run in which a staple code is
absent stops, because the fresh set would then not be the one described here.

In [ ]:
man = D['codigo_act'].str.startswith(('311', '312'))
MAN = (D.loc[man].groupby(['codigo_act', 'nombre_act']).size()
         .rename('records').reset_index().sort_values('records', ascending=False))
print(f'{int(man.sum()):,} records under a food or beverage manufacturing code\n')
if len(MAN):
    print(MAN.to_string(index=False))
    rep.add_table('food and beverage manufacturing codes present', MAN,
                  max_rows=80)

print()
for c_ in STAPLE_CODES:
    n = int((D['codigo_act'] == c_).sum())
    label = D.loc[D['codigo_act'] == c_, 'nombre_act']
    print(f'   {c_}  {n:>8,}  '
          f'{label.iloc[0] if len(label) else "absent"}')

missing = [c for c in STAPLE_CODES if not (D['codigo_act'] == c).any()]
if missing:
    raise AssertionError(
        f'the staple codes {missing} are not in the directory extracts. The '
        'fresh set is defined to include tortilla making and traditional '
        'bakery, so the extracts covering the manufacture of bakery products '
        'and tortillas are needed before this notebook can produce the set it '
        'describes.')

### Codes belonging to no set

A code within these sectors that belongs to no set is listed, since it may be a
source of food the classification does not cover.

In [ ]:
assigned = D[list(LAYERS)].any(axis=1)
UNASSIGNED = (D.loc[~assigned & in_scope].groupby(['codigo_act', 'nombre_act'])
                .size().rename('records').reset_index()
                .sort_values('records', ascending=False))
print(f'{int((~assigned & in_scope).sum()):,} records across '
      f'{len(UNASSIGNED)} codes\n')
print(UNASSIGNED.head(40).to_string(index=False))
rep.add_table('activity codes in no set', UNASSIGNED, max_rows=120)

## 5. Two features of the directory that affect the sets

Stalls inside a public market are generally registered as individual
establishments under the retail food codes, in which case they are already
counted, but the directory records the market in the establishment name rather
than in the activity code.

Warehouse clubs share an activity code with supermarkets and are distinguished
only by name, so the number identified is reported with the names matched.

In [ ]:
print('--- public market stalls ---\n')
mkt = mx.normalise_text(D['nom_estab'].fillna('')).str.contains(
    'MERCADO|TIANGUIS|PUESTO', regex=True)
food = D[FOOD_LAYERS].any(axis=1)
print(f'records naming a market or a stall           {int(mkt.sum()):,}')
print(f'of those carrying a food activity code      {int((mkt & food).sum()):,}')
print(f'of those carrying some other code           '
      f'{int((mkt & ~food).sum()):,}')
print('\nfood codes among the records naming a market or a stall:')
print(D.loc[mkt & food].groupby(['codigo_act', 'nombre_act']).size()
       .rename('records').reset_index().sort_values('records', ascending=False)
       .head(12).to_string(index=False))
print('\nthe other codes, which the name match picks up but no set contains:')
print(D.loc[mkt & ~food].groupby(['codigo_act', 'nombre_act']).size()
       .rename('records').reset_index().sort_values('records', ascending=False)
       .head(8).to_string(index=False))
rep.add_table('codes among records naming a market or a stall',
              D.loc[mkt].groupby(['codigo_act', 'nombre_act']).size()
               .rename('records').reset_index()
               .sort_values('records', ascending=False), max_rows=40)

print('\n--- warehouse clubs ---\n')
print(f'records under the supermarket code           {int(D.supermarket.sum()):,}')
print(f'of which identified as a warehouse club      {int(D.club.sum()):,}')

# A club branch filed under another activity code would not reach the
# supermarket set, so every record carrying a club name is counted and the
# codes they fall under are listed.
CLUB = (D.assign(named=name.str.contains('|'.join(CLUB_NAMES), regex=True))
          .query('named')
          .groupby(['codigo_act', 'nombre_act']).size()
          .rename('records').reset_index()
          .sort_values('records', ascending=False))
print(f'\nrecords carrying a club name, any code         '
      f'{int(CLUB.records.sum()):,}\n')
print(CLUB.to_string(index=False))
rep.add_table('activity codes of records named as a warehouse club', CLUB,
              max_rows=20)

print('\n--- layers entering no set ---\n')
for layer in ['drinking', 'department', 'alcohol_tobacco']:
    print(f'   {layer:16s} {int(D[layer].sum()):>9,}')

## 6. Locating establishments

A record without usable coordinates cannot serve as a destination and is
excluded, with the number reported. The remainder are assigned to a
metropolitan area by falling within its boundary, which is the same boundary the
population blocks were clipped to, so that origins and destinations share one
definition of each study area.

In [ ]:
D = D[D[list(LAYERS)].any(axis=1)].copy()
mx.flow('records in at least one layer', len(D), 'other codes excluded')

good = D['latitud'].between(14, 33) & D['longitud'].between(-118, -86)
print(f'records with usable coordinates: {int(good.sum()):,} of {len(D):,}')
if int((~good).sum()):
    print(f'excluded for missing or implausible coordinates: '
          f'{int((~good).sum()):,}')
D = D[good].copy()
mx.flow('records with usable coordinates', len(D),
        'within the national extent')

# An establishment placed at the centroid of its municipality or locality
# passes a test on the national extent perfectly, so coordinates shared by many
# records are counted. A cluster of destinations on one point makes the distance
# from every block near it wrong, in the direction that favours whichever set
# the clustered records belong to.
key = (D['latitud'].round(5).astype(str) + ','
       + D['longitud'].round(5).astype(str))
sizes = key.value_counts()
print('--- establishments sharing a coordinate ---\n')
for n in (2, 10, 100, 1000):
    members = int(sizes[sizes >= n].sum())
    print(f'   on a point shared by at least {n:>4} records: '
          f'{members:>9,} records ({members / len(D):.2%}) on '
          f'{int((sizes >= n).sum()):,} points')

detail = []
for coord in sizes.head(12).index:
    sel = D[key == coord]
    detail.append({'coordinate': coord, 'records': len(sel),
                   'entities': sel['cve_ent'].nunique(),
                   'municipalities': sel['cve_mun'].nunique(),
                   'commonest code': sel['codigo_act'].mode().iloc[0],
                   'commonest description':
                       str(sel['nombre_act'].mode().iloc[0])[:44]})
CLUST = pd.DataFrame(detail)
print()
print(CLUST.to_string(index=False))
rep.add_table('the largest clusters of shared coordinates', CLUST,
              max_rows=20)

print('\nrecords on a shared point, by layer:\n')
shared = (key.map(sizes) >= 10).to_numpy()
rows = []
for layer in FOOD_LAYERS:
    inset = D[layer].to_numpy()
    if int(inset.sum()):
        rows.append({'layer': layer, 'records': int(inset.sum()),
                     'on a shared point': int((inset & shared).sum()),
                     'share': round(float((inset & shared).sum()
                                          / inset.sum()), 4)})
SHARED = pd.DataFrame(rows)
print(SHARED.to_string(index=False))
rep.add_table('records on a shared coordinate, by layer', SHARED, max_rows=20)

D['shared_point'] = shared
D['point_records'] = key.map(sizes).to_numpy()
print('\nA destination placed at the centroid of its municipality or locality '
      'stands\nat the wrong distance from every block near that centroid. The '
      'flag is\ncarried through to the output so the measures can be recomputed '
      'without\nthose records.')

G = gpd.GeoDataFrame(D, geometry=gpd.points_from_xy(D['longitud'], D['latitud']),
                     crs='EPSG:4326').to_crs(epsg=mx.CRS_METRIC)

G = gpd.sjoin(G, METRO[['slug', 'metro_key', 'tipo', 'geometry']],
              how='inner', predicate='within').drop(columns='index_right')
print(f'\nestablishments within a metropolitan boundary: {len(G):,}')
mx.flow('establishments within a metropolitan boundary', len(G),
        'same boundaries as the population blocks')

## 7. Destination sets

Nine definitions are assembled here, and four more add street commerce to them
at the routing stage. They form two branches and a union.

The self-service branch runs from supermarkets alone, to supermarkets with
warehouse clubs, to those with small format self-service, to those with
pharmacies holding a grocery section. Each step contains the one before it.

The traditional branch runs from the perishable specialists to the fresh and
staple specialists, and from those to two extensions that do not contain one
another: the fresh set with craft dairy, and the fresh set with neighbourhood
grocery stores.

The widest set is the union of every layer that counts as a source of food, so
every other set is contained in it. Within a branch, the distance measured under
a wider definition can never exceed the distance under a narrower one, and that
containment is checked below.

In [ ]:
G['perishable'] = G['codigo_act'].isin(PERISHABLE_CODES)

G['supermarket_only'] = G['supermarket'] & ~G['club']
G['supermarket_club'] = G['supermarket']
G['supermarket_mini'] = G['supermarket'] | G['minisuper']
G['supermarket_mini_pharmacy'] = G['supermarket_mini'] | G['pharmacy_mini']
G['fresh_perishable'] = G['perishable']
G['fresh_only'] = G['fresh']
G['fresh_counter'] = G['fresh'] | G['dairy_craft']
G['fresh_abarrotes'] = G['fresh'] | G['abarrotes']
G['all_food'] = G[FOOD_LAYERS].any(axis=1)

SETS = ['supermarket_only', 'supermarket_club', 'supermarket_mini',
        'supermarket_mini_pharmacy', 'fresh_perishable', 'fresh_only',
        'fresh_counter', 'fresh_abarrotes', 'all_food']

# The branch each set belongs to, and the set it extends, which is what the
# containment checks below rest on.
PARENT = {'supermarket_only': None,
          'supermarket_club': 'supermarket_only',
          'supermarket_mini': 'supermarket_club',
          'supermarket_mini_pharmacy': 'supermarket_mini',
          'fresh_perishable': None,
          'fresh_only': 'fresh_perishable',
          'fresh_counter': 'fresh_only',
          'fresh_abarrotes': 'fresh_only',
          'all_food': None}
BRANCH = {s: ('self-service' if s.startswith('supermarket')
              else 'traditional' if s.startswith('fresh') else 'union')
          for s in SETS}

counts = pd.DataFrame({
    'destination_set': SETS,
    'branch': [BRANCH[s] for s in SETS],
    'extends': [PARENT[s] or '' for s in SETS],
    'establishments': [int(G[s].sum()) for s in SETS],
    'distinct_locations': [
        int(G.loc[G[s], ['latitud', 'longitud']].round(5)
             .drop_duplicates().shape[0]) for s in SETS]})
counts['records_per_location'] = (counts['establishments']
                                  / counts['distinct_locations']).round(2)
counts['relative_to_supermarkets'] = (
    counts['establishments'] / int(G['supermarket_club'].sum())).round(2)
print(counts.to_string(index=False))
rep.add_table('destination set sizes', counts, max_rows=10)

print('\ncomposition of the fresh set:\n')
print(G.loc[G.fresh_only].groupby(['codigo_act', 'nombre_act']).size()
       .rename('establishments').reset_index()
       .sort_values('establishments', ascending=False).to_string(index=False))

print('\ncomposition of the widest set, by layer:\n')
print(pd.Series({l: int(G[l].sum()) for l in FOOD_LAYERS}, name='establishments')
        .to_string())

per = (G.groupby('slug')[SETS].sum().astype(int)
         .sort_values('supermarket_club', ascending=False))
print('\nper metropolitan area, the twenty with the most supermarkets:\n')
print(per.head(20).to_string())
rep.add_table('destination counts by area', per.reset_index(), max_rows=92)

## 8. Checks

Each check names the condition under which it fails.

In [ ]:
print('--- checks ---\n')

print(f'1. duplicate establishment identifiers       '
      f'{int(G["id"].duplicated().sum())}')

# An area with no food establishment at all would be absent from every later
# result, which would be a failure of the spatial join rather than a finding.
empty = sorted(set(METRO['slug']) - set(G.loc[G.all_food, 'slug']))
print(f'2. areas with no food establishment          {len(empty)}')
assert not empty, empty

# An area with no supermarket is possible and is a result rather than an error,
# so it is listed rather than asserted against.
nosup = sorted(set(METRO['slug']) - set(G.loc[G.supermarket_club, 'slug']))
print(f'3. areas with no full format supermarket     {len(nosup)}')
if nosup:
    print('   ', ', '.join(nosup))

# Each set is contained in the set it extends, and every set is contained in
# the widest one. A violation would mean the definitions overlap in a way the
# comparison between them does not allow.
for s, parent in PARENT.items():
    if parent is not None:
        assert (G[parent] <= G[s]).all(), (parent, s)
    assert (G[s] <= G['all_food']).all(), (s, 'all_food')
print('4. containment within each branch and in the widest set  holds')

# The widest definition must equal the sum of its disjoint layers. A difference
# would mean a code had been placed in two of them.
parts = sum(int(G[l].sum()) for l in FOOD_LAYERS)
print(f'5. widest set against the sum of its layers  '
      f'{int(G.all_food.sum()):,} against {parts:,}')
assert int(G.all_food.sum()) == parts

# Bars, cantinas and nightclubs enter no set.
bars = int((G['drinking'] & G[SETS].any(axis=1)).sum())
print(f'6. drinking places inside any set            {bars}')
assert bars == 0

staples = int(G.loc[G.codigo_act.isin(STAPLE_CODES)].shape[0])
print(f'7. tortillerias and bakeries in the fresh set {staples:,}')
assert staples > 0

print(f'\n8. supermarkets and warehouse clubs          '
      f'{int(G.supermarket_club.sum()):,}')
print(f'9. specialist fresh and staple sellers       {int(G.fresh_only.sum()):,}')
print(f'10. neighbourhood grocery stores             {int(G.abarrotes.sum()):,}')
print(f'11. small format self-service                {int(G.minisuper.sum()):,}')
print(f'12. pharmacies with a grocery section        {int(G.pharmacy_mini.sum()):,}')
print(f'13. craft dairy producers                    {int(G.dairy_craft.sum()):,}')
print(f'14. remaining food retail                    {int(G.other_food.sum()):,}')
print(f'15. prepared food outlets                    {int(G.prepared.sum()):,}')
print(f'16. drinking places, in no set               {int(G.drinking.sum()):,}')
print(f'17. every food source in the register        {int(G.all_food.sum()):,}')

## 9. Write results

In [ ]:
cols = (['id', 'nom_estab', 'codigo_act', 'nombre_act', 'per_ocu',
         'slug', 'metro_key', 'tipo', 'latitud', 'longitud',
         'shared_point', 'point_records']
        + list(LAYERS) + ['club', 'perishable'] + SETS)
df = pd.DataFrame(G.drop(columns='geometry'))[cols]
df['x_m'] = G.geometry.x.values
df['y_m'] = G.geometry.y.values

out = os.path.join(mx.DIRS['results'], '05_outlets.parquet')
df.to_parquet(out, index=False)
print('written:', out, f'({os.path.getsize(out) / 1e6:.1f} MB)')

print()
print(mx.flow_table().to_string(index=False))
mx.save_flow(os.path.join(mx.DIRS['logs'], '05_sample_flow.csv'))
rep.add_table('attrition', mx.flow_table(), max_rows=20)

rep.note(f'{len(G):,} establishments located; '
         f'{int(G.supermarket_club.sum()):,} supermarkets and warehouse clubs; '
         f'{int(G.fresh_only.sum()):,} specialist fresh and staple sellers; '
         f'{int(G.abarrotes.sum()):,} neighbourhood grocery stores; '
         f'{int(G.all_food.sum()):,} in the widest set, which contains every '
         f'other set; {int(G.drinking.sum()):,} bars, cantinas and nightclubs '
         f'in no set')
rep.save()